In [15]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver

Get:1 https://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Ign:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy Release
Get:10 https://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,093 B]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:14 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 2,918 B in 2s (1,875 B/s)
Reading package lists... Done
W: Skipping

In [26]:
import csv
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException, TimeoutException, NoSuchElementException, ElementClickInterceptedException
)
from selenium.webdriver.chrome.options import Options
from google.colab import drive
from webdriver_manager.chrome import ChromeDriverManager

# Montagem do Google Drive para salvar os dados
drive.mount('/content/drive')

# Definir o caminho para salvar o arquivo CSV na pasta "output" no Google Drive
base_dir = '/content/drive/MyDrive'
output_dir = os.path.join(base_dir, 'output')

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'tumelero_produtos.csv')

# Configuração do WebDriver para uso no Colab
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Atualizar o chromedriver
service = ChromeService(ChromeDriverManager().install())

# Inicializa o WebDriver
driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    # Abre a página do Tumelero
    driver.get('https://www.tumelero.com.br/pisos-e-revestimentos/pisos/pisos-ceramicos')

    # Espera até que o contêiner de produtos esteja visível
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, '#vitrine-products-smarthint .div-products-smarthint-created'))
    )

    # Função para fechar possíveis overlays
    def fechar_overlay():
        try:
            overlays = driver.find_elements(By.CSS_SELECTOR, 'div.onetrust-pc-dark-filter')
            for overlay in overlays:
                driver.execute_script("arguments[0].click();", overlay)
                time.sleep(1)  # Aguarda a remoção do overlay
        except NoSuchElementException:
            print("Nenhum overlay encontrado.")

    # Função para rolar até o final da página
    def scroll_to_bottom():
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)  # Aguarda o carregamento dos novos produtos

    # Função para extrair dados da página
    def extrair_dados():
        dados = []
        pagina_atual = 1

        while True:
            try:
                # Encontrar a lista de produtos dentro do contêiner
                lista_produtos = driver.find_element(By.CSS_SELECTOR, '#vitrine-products-smarthint .div-products-smarthint-created')
                produtos = lista_produtos.find_elements(By.CSS_SELECTOR, 'li[data-smarthintfeature="search"]')
                print(f"Total de produtos encontrados na página {pagina_atual}: {len(produtos)}")

                for produto in produtos:
                    try:
                        nome = produto.find_element(By.CSS_SELECTOR, 'h2.x-shelf__title').text
                        preco = produto.find_element(By.CSS_SELECTOR, 'span.x-shelf__best-price strong').text
                        imagem = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container img').get_attribute('src')
                        link = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container a').get_attribute('href')

                        dados.append({
                            'nome': nome,
                            'preco': preco,
                            'imagem': imagem,
                            'link': link
                        })
                    except NoSuchElementException:
                        print("Elemento não encontrado dentro do produto. Ignorando este produto.")
                        continue

                try:
                    fechar_overlay()

                    # Rola para baixo e tenta encontrar o botão de carregamento
                    scroll_to_bottom()

                    try:
                        botao_mais = WebDriverWait(driver, 30).until(
                            EC.presence_of_element_located((By.CSS_SELECTOR, 'div.father-loadmore-smarthint button.cf-load-more'))
                        )
                        driver.execute_script("arguments[0].scrollIntoView(true);", botao_mais)  # Scroll para o botão

                        # Tentar clicar com retry
                        for _ in range(3):
                            try:
                                WebDriverWait(driver, 10).until(
                                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.father-loadmore-smarthint button.cf-load-more'))
                                ).click()
                                time.sleep(2)  # Espera o carregamento dos produtos
                                break
                            except ElementClickInterceptedException:
                                print("Overlay ou outro elemento interceptou o clique. Tentando novamente...")
                                time.sleep(2)
                        else:
                            print("Não foi possível clicar no botão 'Mais produtos' após várias tentativas.")
                            break

                        pagina_atual += 1
                        print(f"Próxima página: {pagina_atual}")

                    except NoSuchElementException:
                        print("Não foi possível encontrar o botão 'Mais produtos'. Tentando outro tipo de botão de carregamento.")
                        try:
                            botao_carregar = WebDriverWait(driver, 30).until(
                                EC.presence_of_element_located((By.CSS_SELECTOR, 'div.father-loading-vitrine-smarthint'))
                            )
                            driver.execute_script("arguments[0].scrollIntoView(true);", botao_carregar)  # Scroll para o botão
                            time.sleep(2)  # Espera o carregamento dos produtos
                        except NoSuchElementException:
                            print("Não foi possível encontrar o botão de carregamento alternativo.")
                            break
                        except StaleElementReferenceException:
                            print("O botão de carregamento alternativo não está mais disponível. Encerrando coleta.")
                            break

                except StaleElementReferenceException:
                    print("O botão de carregamento não está mais disponível. Encerrando coleta.")
                    break

            except NoSuchElementException as e:
                print(f"Erro ao encontrar produtos: {e}")
                break

        return dados

    # Execução principal
    dados = extrair_dados()

    # Verifica se dados foram coletados
    if not dados:
        print("Nenhum dado coletado.")
    else:
        # Exibe os dados coletados no console
        for dado in dados:
            print(f"Nome: {dado['nome']}, Preço: {dado['preco']}, Imagem: {dado['imagem']}, Link: {dado['link']}")

        # Salva os dados no arquivo CSV
        with open(output_path, 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=['nome', 'preco', 'imagem', 'link'])
            writer.writeheader()
            writer.writerows(dados)
        print(f"Dados salvos em {output_path}")

finally:
    # Fechando o navegador
    driver.quit()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de produtos encontrados na página 1: 36
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Não foi possível clicar no botão 'Mais produtos' após várias tentativas.
Nome: Piso Cerâmico Stone HD MRE 50,5X50,5cm Cinza Pisoforte, Preço: R$ 22,40/m², Imagem: https://tumelerosaintgobain.vteximg.com.br/arquivos/ids/194516-200-200/PISO-CERAMICO-STONE-HD-MRE-505X505-CINZA-PISOFORTE-10233330.jpg?v=637922340338800000, Link: https://www.tumelero.com.br/piso-ceramico-stone-hd-mre-50-5x50-5-cinza-pisoforte-10233330/p
Nome: Piso Cerâmico Elemento Cinza Plus HD 61X61Cm Retificado Caixa 2,23M2 Cecafi, Preço: R$ 33,20/m², Imagem: https://tumelerosaintgobain.vteximg.com.br/arquivos/ids/187129-200-200/888800779.jpg?v